In [1]:
#
# geometric non-linear elasticity with Neo-Hooke hyperelastic material
#
# featuring automatic differentiation in SymbolicEnergy
#

import netgen.geom2d as geom2d
from ngsolve import *
from ngsolve.webgui import Draw
geo = geom2d.SplineGeometry()
pnums = [ geo.AddPoint (x,y,maxh=0.01) for x,y in [(0,0), (1,0), (1,0.1), (0,0.1)] ]
for p1,p2,bc in [(0,1,"bot"), (1,2,"right"), (2,3,"top"), (3,0,"left")]:
     geo.Append(["line", pnums[p1], pnums[p2]], bc=bc)
mesh = Mesh(geo.GenerateMesh(maxh=0.05))
Draw(mesh)

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

BaseWebGuiScene

In [2]:
E, nu = 210e-3, 0.45
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

fes = H1(mesh, order=2, dirichlet="left", dim=mesh.dim)
# fes = VectorH1(mesh, order=2, dirichlet="left")

u  = fes.TrialFunction()

force = CoefficientFunction( (0,1e-4) )

I = Id(mesh.dim)
F = I + Grad(u)
C = F.trans * F
E = 0.5 * (C-I)

def Pow(a, b):
    return a**b  # exp (log(a)*b)
  
def NeoHooke (C):
    return 0.5 * mu * (Trace(C-I) + 2*mu/lam * Pow(Det(C),-lam/2/mu) - 1)



factor = Parameter(1)

a = BilinearForm(fes, symmetric=False)
a += Variation (NeoHooke(C).Compile()*dx)
a += Variation ((-factor * InnerProduct(force,u) ).Compile()*dx)


u = GridFunction(fes)
u.vec[:] = 0

res = u.vec.CreateVector()
w = u.vec.CreateVector()


for loadstep in range(10):
    
    print ("loadstep", loadstep)
    factor.Set (loadstep+1)
    
    for it in range(5):
        print ("Newton iteration", it)
        print ("energy = ", a.Energy(u.vec))
        a.Apply(u.vec, res)
        a.AssembleLinearization(u.vec)
        inv = a.mat.Inverse(fes.FreeDofs() ) 
        w.data = inv*res
        print ("err^2 = ", InnerProduct (w,res))
        u.vec.data -= w
    
    Draw (u, mesh, deformation=True, scale=1)
    SetVisualization (deformation=True)
    input ("<press a key>")
    

loadstep 0
Newton iteration 0
energy =  -0.002816091954022989
err^2 =  2.28576919087418e-07
Newton iteration 1
energy =  -0.002815597782171912
err^2 =  1.2362321630305044e-06
Newton iteration 2
energy =  -0.0028162057448939347
err^2 =  7.432847610000193e-10
Newton iteration 3
energy =  -0.002816206116694464
err^2 =  7.06819080925289e-16
Newton iteration 4
energy =  -0.0028162061166948182
err^2 =  2.288190161009316e-23


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

<press a key> 


loadstep 1
Newton iteration 0
energy =  -0.0028164341915028358
err^2 =  2.2707581152991177e-07
Newton iteration 1
energy =  -0.0028159517261674883
err^2 =  1.2097043997402568e-06
Newton iteration 2
energy =  -0.0028165467564963335
err^2 =  7.182583843268052e-10
Newton iteration 3
energy =  -0.002816547114147082
err^2 =  1.047172231508963e-12
Newton iteration 4
energy =  -0.002816547114670922
err^2 =  4.045795527540424e-17


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

...

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

<press a key> 


loadstep 9
Newton iteration 0
energy =  -0.00282644848118179
err^2 =  1.5040402238562822e-07
Newton iteration 1
energy =  -0.002826367709281634
err^2 =  3.1060557544837524e-07
Newton iteration 2
energy =  -0.0028265217866156054
err^2 =  1.2311350581172327e-10
Newton iteration 3
energy =  -0.002826521845470132
err^2 =  4.906642417482229e-13
Newton iteration 4
energy =  -0.0028265218457161024
err^2 =  7.560381067175483e-18


WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

<press a key> 
